In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, os, warnings
warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [2]:
CSV_PATH = '/home/chinmaya/Desktop/coding/aiagent/health_care/data sets/heart /heart.csv'  # update to your path
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
df.head()

Shape: (1025, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [3]:
def engineer_features(df):
    df = df.copy()
    df['age_thalach_ratio'] = df['age'] / (df['thalach'] + 1)
    df['chol_age_ratio']    = df['chol'] / (df['age'] + 1)
    df['oldpeak_cat'] = pd.cut(
        df['oldpeak'], bins=[-0.1, 0.5, 2.0, 6.5], labels=[0, 1, 2]
    ).astype(int)
    df['age_hr_risk'] = ((df['age'] > 55) & (df['thalach'] < 140)).astype(int)
    return df

print('engineer_features defined')

engineer_features defined


In [4]:
FEATURES = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal',
    'age_thalach_ratio', 'oldpeak_cat', 'age_hr_risk', 'chol_age_ratio'
]
TARGET = 'target'
print(f'Feature count: {len(FEATURES)}')

Feature count: 17


In [5]:
def build_pipeline(clf):
    return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

print('build_pipeline defined')

build_pipeline defined


In [6]:
# ── BUG 1 FIX: return calibrated model, not best_pipeline ─────
# ── BUG 3 FIX: CalibratedClassifierCV now imported in Cell 1 ──
# ── BUG 5 FIX: stratify=y added to train_test_split ───────────
def train_and_evaluate(df):
    df = engineer_features(df)
    X  = df[FEATURES]
    y  = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    candidates = {
        'Logistic Regression': LogisticRegression(max_iter=1000, C=0.5),
        'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
        'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    best_name, best_pipe_obj, best_auc = None, None, 0

    print('── Cross-Validation Results ──')
    for name, clf in candidates.items():
        pipe     = build_pipeline(clf)
        scores   = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
        mean_auc = scores.mean()
        print(f'  {name:<25}  AUC = {mean_auc:.3f} +/- {scores.std():.3f}')
        if mean_auc > best_auc:
            best_auc = mean_auc
            best_name = name
            best_pipe_obj = pipe

    print(f'Best: {best_name}  (AUC = {best_auc:.3f})')

    calibrated = CalibratedClassifierCV(best_pipe_obj, method='isotonic', cv=5)
    calibrated.fit(X_train, y_train)

    y_pred  = calibrated.predict(X_test)
    y_proba = calibrated.predict_proba(X_test)[:, 1]

    print('\n── Test Set Report ──')
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}')

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=['No Disease', 'Heart Disease'],
        cmap='Blues', ax=ax
    )
    ax.set_title('Heart Disease Confusion Matrix')
    plt.tight_layout()
    plt.savefig('heart_confusion_matrix.png', dpi=100)
    plt.show()

    # BUG 1 FIX: return calibrated (fitted + probability-calibrated)
    # NOT best_pipe_obj (unfitted skeleton — has no predict_proba yet)
    return calibrated, X_train, X_test, y_train, y_test

print('train_and_evaluate defined')

train_and_evaluate defined


In [7]:
# ── BUG 2 FIX: Call train_and_evaluate NOW and capture return values ──
# Original Cell 8 called save_model(best_pipeline) but best_pipeline
# only ever existed inside train_and_evaluate — never in global scope.
# Solution: call the function here and assign the returned model to 'model'.

model, X_train, X_test, y_train, y_test = train_and_evaluate(df)
print(f'model ready: {type(model).__name__}')

── Cross-Validation Results ──
  Logistic Regression        AUC = 0.917 +/- 0.009
  Random Forest              AUC = 0.992 +/- 0.005
  Gradient Boosting          AUC = 0.993 +/- 0.009
Best: Gradient Boosting  (AUC = 0.993)

── Test Set Report ──
              precision    recall  f1-score   support

  No Disease       1.00      1.00      1.00       100
     Disease       1.00      1.00      1.00       105

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205

ROC-AUC: 1.000
model ready: CalibratedClassifierCV


In [8]:
def plot_feature_importance(calibrated_model, feature_names):
    try:
        base        = calibrated_model.calibrated_classifiers_[0].estimator
        importances = base.named_steps['clf'].feature_importances_
        fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        fi_df = fi_df.sort_values('importance', ascending=True)

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh(fi_df['feature'], fi_df['importance'], color='steelblue')
        ax.set_title('Heart Disease Feature Importance')
        ax.set_xlabel('Importance score')
        plt.tight_layout()
        plt.savefig('heart_feature_importance.png', dpi=100)
        plt.show()

        print('Top 5 features:')
        print(fi_df.tail(5)[['feature', 'importance']].to_string(index=False))
    except Exception as e:
        print(f'Feature importance not available: {e}')

# Call directly — do NOT write: model = plot_feature_importance(...)
# That would overwrite model with None (function returns nothing)
plot_feature_importance(model, FEATURES)

Top 5 features:
feature  importance
    age    0.072927
   thal    0.085269
     ca    0.116589
oldpeak    0.125329
     cp    0.272842


In [9]:
def save_model(mdl, path='models/heart_disease_model.pkl'):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    joblib.dump(mdl, path)
    print(f'Saved: {path}')

# BUG 2 FIX: pass 'model' (created in Cell 7) not 'best_pipeline'
# best_pipeline was a local variable inside train_and_evaluate — never global
save_model(model)

Saved: models/heart_disease_model.pkl


In [10]:
def predict_heart_disease(mdl, patient_data: dict) -> dict:
    df_p = pd.DataFrame([patient_data])
    df_p = engineer_features(df_p)
    for col in FEATURES:
        if col not in df_p.columns:
            df_p[col] = 0

    prob = float(mdl.predict_proba(df_p[FEATURES])[0][1])
    risk = 'High' if prob >= 0.70 else 'Moderate' if prob >= 0.40 else 'Low'

    return {
        'disease':     'Heart Disease',
        'probability': round(prob, 3),
        'risk_level':  risk,
        'confidence':  f'{prob * 100:.1f}%'
    }

print('predict_heart_disease defined')

predict_heart_disease defined


In [11]:


sample_patient = {
    'age': 62, 'sex': 1, 'cp': 2, 'trestbps': 150, 'chol': 260, 'fbs': 1,
    'restecg': 1, 'thalach': 140, 'exang': 1, 'oldpeak': 2.5,
    'slope': 2, 'ca': 2, 'thal': 2
}

result = predict_heart_disease(model, sample_patient)
print('── Sample Prediction ──')
for k, v in result.items():
    print(f'  {k}: {v}')

print('\nDISCLAIMER: This is a predictive model only, not a medical diagnosis.')

── Sample Prediction ──
  disease: Heart Disease
  probability: 0.133
  risk_level: Low
  confidence: 13.3%

DISCLAIMER: This is a predictive model only, not a medical diagnosis.
